# Cohen 2020：原作者评估代码的兼容复现

目标：完整复现 Table 1，理解 94 张胸片、65 位患者、两种目标与五种特征方案。

原始 notebook 保存在 `sources/covid-severity__xray_covid_eval.ipynb`。下面的 `evaluate` 函数逐字保留原作者第 33 个 cell（从 0 计数）的代码；运行路径、特征加载与结果收集兼容现代 Python。原 notebook 的 opacity 汇总被注释，这里对两个目标分别执行。

先运行 `python reproduce.py prepare` 和 `python reproduce.py run`。选择项目 `.venv` 的 Python 解释器。所有已有单元格输出都在本机真实执行生成。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'reproduce.py').exists():
    ROOT = ROOT / 'covid_severity_reproduction'
assert (ROOT / 'reproduce.py').exists(), '请从项目目录打开本 notebook'
sys.path.insert(0, str(ROOT))
import reproduce as repro
import numpy as np
import pandas as pd
import torch
from threadpoolctl import threadpool_limits
torch.set_num_threads(4)
cohort = pd.read_csv(ROOT / 'results/cohort.csv')
cache = np.load(ROOT / 'results/features.npz', allow_pickle=False)
d_feats, d_preds = cache['features'], cache['logits']
assert len(cohort) == 94 and cohort.patientid.nunique() == 65
assert d_feats.shape == (94, 1024) and d_preds.shape == (94, 18)
print('数据:', len(cohort), '张胸片 /', cohort.patientid.nunique(), '位患者')
print('冻结特征:', d_feats.shape, '原始 logits:', d_preds.shape)
print(cohort[['geographic_mean', 'opacity_mean']].describe().round(2).to_string())
import matplotlib.pyplot as plt


数据: 94 张胸片 / 65 位患者
冻结特征: (94, 1024) 原始 logits: (94, 18)
       geographic_mean  opacity_mean
count            94.00         94.00
mean              3.34          2.46
std               2.37          1.52
min               0.00          0.00
25%               1.30          1.30
50%               3.00          2.30
75%               4.92          3.70
max               8.00          6.00


## 1. 评分、特征和预测分别是什么

- `opacity_mean`：三位专家的不透明度评分均值，0–6；`geographic_mean`：病变范围评分均值，0–8。两者各训练一个回归模型。
- DenseNet 已在非 COVID 胸片上预训练，本实验冻结其参数。1024 维特征取最后卷积层经过 ReLU 和全局均值池化的向量；18 维 logits 取线性分类头、sigmoid 之前的输出。
- 单特征模型使用 `Lung Opacity` logit。它是 DenseNet 的诊断输出特征，**不是专家的 opacity 标签**。线性回归再将这个特征映射成专家的严重度分数。
- 四特征为 Lung Opacity、Pneumonia、Infiltration、Consolidation。常数基线预测训练集评分均值。

预处理使用历史代码规则：取图像第一通道，按 255 映射至 [-1024,1024]，中心裁成正方形，缩放至 224×224。无增强、特征标准化、正则化或预测裁剪。

## 2. 原作者的评估函数

`GroupShuffleSplit` 按患者划分，避免同一患者的不同时间点同时进入训练集和测试集。50/50 指患者比例，图像数量可以不同。下面原始函数中的额外 ACC/解释方差字段不用于 Table 1。

In [2]:
import sklearn, sklearn.linear_model, sklearn.metrics, sklearn.model_selection
import scipy, scipy.stats
def evaluate(data, labels, title, plot=True, groups=None, seed=0, method="mse", target_str=""):
    X = data
    y = labels.astype(float)
    
    gss = sklearn.model_selection.GroupShuffleSplit(train_size=0.5,test_size=0.5, random_state=seed)
    train_inds, test_inds = next(gss.split(X, y, groups))
    X_train, X_test, y_train, y_test = X[train_inds], X[test_inds], y.iloc[train_inds], y.iloc[test_inds]
    #print("X_train", X_train.shape, "X_test", X_test.shape)

    if method=="mse":
        model = sklearn.linear_model.LinearRegression()
    elif method=="huber":
        model = sklearn.linear_model.HuberRegressor()
    elif method=="mae":
        # loss = max(0, |y - p| - epsilon)
        model = sklearn.linear_model.SGDRegressor(loss="epsilon_insensitive", epsilon=0, l1_ratio=1, random_state=seed)
    
    model = model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    pdiff = y_test - y_pred
    
    res = {}
    res["name"] = title
    res["R^2"] = sklearn.metrics.r2_score(y_test, y_pred)
    res["Correlation"] = scipy.stats.pearsonr(y_pred,y_test)[0]
    res["< 0.7 ACC"] = ((y_pred - y_test) < 0.7).sum()/len(y_test)
    res["explained_variance"] = sklearn.metrics.explained_variance_score(y_test, y_pred)
    res["MAE"] = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    res["MSE"] = sklearn.metrics.mean_squared_error(y_test, y_pred)
    res["method"] = method
    res["# params"] = "{}+1".format(model.coef_.shape[0])

    
    if plot:
        fig, ax = plt.subplots(figsize=(6,4), dpi=100)
        for x,y,yp in zip(y_test,y_test,y_pred):
            plt.plot((x,x),(y,yp),color='red',marker='')

        pmax = int(np.max([y_pred.max(), y_test.max()]))+2
        plt.plot(range(pmax),range(pmax), c="gray", linestyle="--")
        plt.xlim(0,pmax-1)
        plt.ylim(0,pmax-1)

        plt.scatter(y_test, y_pred);
        plt.ylabel("Model prediction ($y_{pred}$)")
        plt.xlabel("Ground Truth ($y_{true}$)")
        plt.title(title);
        plt.text(0.01,0.97, "$R^2$={0:0.2f}".format(res["R^2"])+ "\n"+ 
                 "Correlation={0:0.2f}".format(res["Correlation"]), ha='left', va='top', transform=ax.transAxes)

    return res#, pdiff


## 3. 对两个评分、五种输入各重复 50 次

每个 seed 下，各方法和两个目标共用相同患者划分。现代 pandas 已移除 `DataFrame.append`，这里用列表收集后统一创建 DataFrame。常数预测的 Pearson 在数学上未定义，现代 SciPy 返回 NaN；为了对照论文，明确将常数基线的展示值设为 0。

In [3]:
import warnings
from scipy.stats import ConstantInputWarning
warnings.filterwarnings('ignore', category=ConstantInputWarning)
feature_sets = [
    ('lung_opacity', d_preds[:, [16]]),
    ('four_logits', d_preds[:, [16, 8, 2, 1]]),
    ('eighteen_logits', d_preds),
    ('intermediate_features', d_feats),
    ('constant', np.zeros((94, 1))),
]
records = []
with threadpool_limits(limits=4):
    for target, column in [('opacity', 'opacity_mean'), ('extent', 'geographic_mean')]:
        for title, inputs in feature_sets:
            for seed in range(50):
                result = evaluate(inputs, cohort[column], title, plot=False,
                                  groups=cohort.patientid, seed=seed, method='mse')
                if title == 'constant':
                    result['Correlation'] = 0.0  # 论文的展示约定
                result.update(target=target, seed=seed)
                records.append(result)
results = pd.DataFrame(records)
results.to_csv(ROOT / 'results/author_notebook_metrics.csv', index=False)
summary = results.groupby(['target', 'name'])[['Correlation', 'R^2', 'MAE', 'MSE']].agg(['mean', 'std'])
summary.to_csv(ROOT / 'results/author_notebook_table1.csv')
print('完成', len(results), '次拟合；以下为均值和样本标准差 (ddof=1)')
print(summary.round(2).to_string())

完成 500 次拟合；以下为均值和样本标准差 (ddof=1)
                              Correlation         R^2         MAE         MSE      
                                     mean   std  mean   std  mean   std  mean   std
target  name                                                                       
extent  constant                     0.00  0.00 -0.09  0.09  2.07  0.17  5.97  0.90
        eighteen_logits              0.77  0.06  0.50  0.19  1.31  0.20  2.72  0.92
        four_logits                  0.81  0.03  0.61  0.07  1.17  0.12  2.12  0.37
        intermediate_features        0.74  0.08  0.47  0.15  1.34  0.13  2.86  0.60
        lung_opacity                 0.81  0.04  0.60  0.08  1.16  0.10  2.15  0.33
opacity constant                     0.00  0.00 -0.08  0.10  1.29  0.10  2.40  0.37
        eighteen_logits              0.73  0.06  0.42  0.16  0.90  0.12  1.28  0.34
        four_logits                  0.80  0.04  0.60  0.07  0.76  0.07  0.88  0.15
        intermediate_features        0.68  0

## 4. 核对脚本和原作者函数结果

该检查对全部 500 次拟合的四项指标逐项比较，避免只有汇总均值相近而底层评估协议不同。数据匹配、图像校验与每个患者划分的无泄漏检查在 `reproduce.py run` 中执行。

In [4]:
script_results = pd.read_csv(ROOT / 'results/metrics_per_split.csv').sort_values(['target', 'method', 'seed'])
notebook_results = results.drop(columns='method').rename(columns={'name': 'method', 'Correlation': 'pearson_r', 'R^2': 'r2', 'MAE': 'mae', 'MSE': 'mse'}).sort_values(['target', 'method', 'seed'])
np.testing.assert_allclose(script_results[repro.METRICS].to_numpy(), notebook_results[repro.METRICS].to_numpy(), rtol=1e-7, atol=1e-7)
print('PASS：原作者 evaluate 函数与复现脚本的 500×4 项指标一致。')
comparison = pd.read_csv(ROOT / 'results/table1_comparison.csv')
print(comparison[['target', 'method', 'mae_mean', 'mae_paper_mean', 'mae_delta']].round(3).to_string(index=False))

PASS：原作者 evaluate 函数与复现脚本的 500×4 项指标一致。
 target                method  mae_mean  mae_paper_mean  mae_delta
opacity          lung_opacity     0.788            0.78      0.008
opacity           four_logits     0.761            0.76      0.001
opacity       eighteen_logits     0.897            0.86      0.037
opacity intermediate_features     1.012            1.01      0.002
opacity              constant     1.289            1.24      0.049
 extent          lung_opacity     1.161            1.14      0.021
 extent           four_logits     1.172            1.19     -0.018
 extent       eighteen_logits     1.307            1.32     -0.013
 extent intermediate_features     1.341            1.36     -0.019
 extent              constant     2.072            2.00      0.072


## 5. 汇报时怎样解释

本次支持原文的主要趋势：1/4 个相关 logits 比 18 个输出或 1024 维特征具有更好的平均 MAE；参数更多并不保证小样本上的泛化更好。四特征在 opacity 上的 MAE 最低，单特征的参数更少，两者权衡都应如实展示。

数字接近但未逐位一致。原始 notebook 的已保存 extent 结果本身也与论文表格略有差异。论文未固定数据 commit，本次选用了满足原 notebook 199 PA / 94 带评分图像的历史快照；现代数值库和图像处理库也可能影响结果。50 次划分共享样本，标准差表示划分波动，不是置信区间，也不能直接据此声称某方案统计显著更优。

本次完成 Part I 的 Table 1 实验，包含一组固定 seed 的定性散点图；未复现全文 t-SNE/梯度可视化。课程 Part II 仍需要新的方法对照和课堂 checklist 的批判性评价。RALO 可作为后续外部验证，但其 opacity 量表不完全相同。

完整操作与数据来源见 `README.md`，图表见 `results/mae_comparison.png` 和 `results/heldout_scatter.png`。